<a href="https://colab.research.google.com/github/pranavkarnik/de-practice/blob/main/retail_db/file_format_converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!git clone https://github.com/pranavkarnik/de-practice/

Cloning into 'de-practice'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 31 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 3.37 MiB | 5.35 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [ ]:
import glob
import os
import json
import re
import pandas as pd

In [19]:
# Define the base location and the regex pattern for splitting file paths
split_key = '[/\\\\]'
base_location = '/content/de-practice'

In [4]:
# Get column names from the schema for a given dataset name and sort them by the specified sorting key
def get_column_names(schemas, ds_name, sorting_key='column_position'):
    column_details = schemas[ds_name]
    columns = sorted(column_details, key=lambda col: col[sorting_key])
    return [col['column_name'] for col in columns]

In [5]:
# Read a CSV file and return a pandas DataFrame with the appropriate column names based on the schema
def read_csv(file, schemas):
    file_path_list = re.split(split_key, file)
    ds_name = file_path_list[-2]
    file_name = file_path_list[-1]
    columns = get_column_names(schemas, ds_name)
    df = pd.read_csv(file, names=columns)
    return df

In [8]:
# Convert dataframe to json and save it to the target directory
def to_json(df, tgt_base_dir, ds_name, file_name):
    json_file_path = f'{tgt_base_dir}/{ds_name}/{file_name}.json'
    os.makedirs(f'{tgt_base_dir}/{ds_name}', exist_ok=True)
    df.to_json(
        json_file_path,
        orient='records',
        lines=True
    )

In [21]:
# Convert all files in the source directory to json and save them to the target directory
def file_converter(src_base_dir, tgt_base_dir, ds_name):
    schemas = json.load(open(f'{src_base_dir}/schemas.json'))
    files = glob.glob(f'{src_base_dir}/{ds_name}/part-*')

    for file in files:
        df = read_csv(file, schemas)
        file_name = re.split(split_key, file)[-1]
        to_json(df, tgt_base_dir, ds_name, file_name)

In [18]:
# Process all files in the source directory and save them to the target directory
import os
def process_files(ds_names=None):
    src_base_dir = f'{base_location}/retail_db'
    tgt_base_dir = f'{base_location}/output'
    print(os.getcwd())
    print(f'${src_base_dir}')
    print(f'${tgt_base_dir}')
    schemas = json.load(open(f'{src_base_dir}/schemas.json'))
    if not ds_names:
        ds_names = schemas.keys()
    for ds_name in ds_names:
        print(f'Processing {ds_name}')
        file_converter(src_base_dir, tgt_base_dir, ds_name)

In [23]:
ds_name = 'orders'

process_files()

/content
$/content/de-practice/retail_db
$/content/de-practice/output
Processing departments
Processing categories
Processing orders
Processing products
Processing customers
Processing order_items


In [ ]:
# Run the file processing function if this script is executed directly
if __name__ == "__main__":
    process_files()